# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of Analysis + Time Window

### Unit of Analysis

The unit of analysis is **one anonymized content page (content item)**.

Each row represents a single content page together with its search performance, engagement, freshness, and content-related metrics. The page is identified by a unique `content_id`, while `client_id` identifies the client that owns the page.

### Time Window

Most performance metrics summarize activity over a **trailing 90-day period**, including impressions, clicks, sessions, engagement, and AI traffic.

The dataset also contains two additional 30-day windows:

- `*_last_30d` → the most recent 30 days
- `*_prev_30d` → the 30 days immediately before that

These windows allow us to compare recent performance against an earlier period and observe trends. They are useful for understanding changes over time while avoiding data leakage when building machine learning models.

In [2]:
from pathlib import Path
import pandas as pd
import os

if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/hafizahmadadilaiengineer/flyrank-ml-internship.git

repo_root = Path("flyrank-ml-internship")

df = pd.read_csv(repo_root / "data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)

print("\nUnique Content Pages:", df["content_id"].nunique())

print("Unique Clients:", df["client_id"].nunique())

display(df[[
    "content_id",
    "client_id",
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d"
]].head())

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 117 (delta 34), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 1.84 MiB | 11.73 MiB/s, done.
Resolving deltas: 100% (34/34), done.
Dataset Shape: (30000, 44)

Unique Content Pages: 30000
Unique Clients: 32


,content_id,client_id,impressions_90d,impressions_last_30d,impressions_prev_30d
0,content_304f48230142,client_f369cb89fc,3803,578,987
1,content_a1fb4e703a9e,client_4e07408562,15320,2501,5915
2,content_9aa793d4d895,client_7f2253d7e2,12581,2382,6089
3,content_331d6c4de07b,client_19581e27de,11751,3626,4206
4,content_d99b7a2d90ca,client_3fdba35f04,19140,4211,6452


## 2. Fields: Feature / Label / Context / Excluded

### Features

The feature set consists of observable information that would be available before making a business decision.

Examples include:

- search_volume
- competition
- cpc
- content_type
- main_intent
- word_count
- char_count
- impressions_90d
- clicks_90d
- sessions_90d
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct
- content_age_days
- days_since_last_update

### Label / Target

For future modeling, the outcome relates to identifying content that should be prioritized## Self-check

- [x] I defined the unit of analysis.
- [x] I described the time window.
- [x] I separated features, labels, context, and excluded fields.
- [x] I verified my assumptions using code.
- [x] I documented important limitations of the dataset.
- [x] I explained why some fields should not be used for modeling.
 for refresh.

Although the dataset contains `trend_direction` and `trend_pct`, these columns are derived from observed performance and should not be used as input features because they introduce data leakage.

### Context Fields

These fields provide context but should not be used as predictive features.

Examples:

- content_id
- client_id
- provider_used
- model_used

### Excluded Fields

The following fields should be excluded during model training:

- trend_direction
- trend_pct

These fields are derived from the outcome and would leak future information into the model.

In [3]:
print("Columns")

for col in df.columns:
    print(col)

Columns
content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct


## 3. Verify the Data Contract

The assumptions in the data contract should be verified with simple queries rather than accepted without evidence.

The following checks confirm:

- one row represents one content page
- the dataset contains expected identifiers
- missing values exist in some columns and should be handled carefully
- the time-window columns are available for trend analysis

In [4]:
print("Duplicate content_id:", df["content_id"].duplicated().sum())

print("\nMissing Values")

missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

display(missing[missing > 0])

print("\nTime Window Columns")

display(df.filter(regex="90d|last_30d|prev_30d").head())

Duplicate content_id: 0

Missing Values


,0
provider_used,21438
word_count,7699
char_count,7699
word_count_tier,7699
char_count_tier,7699
model_used,5733
trend_pct,3388
competition_level,2610
search_volume,2468
cpc,2468



Time Window Columns


,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d
0,3803,29,22,17,16,1,0,1,578,2,2,987,13,9
1,15320,7,10,9,9,0,0,1,2501,2,3,5915,1,2
2,12581,11,14,11,11,0,0,4,2382,1,1,6089,3,3
3,11751,58,87,78,75,1,0,3,3626,22,35,4206,17,26
4,19140,24,177,145,144,0,0,43,4211,10,14,6452,2,9


## 4. Data Limits

This starter dataset is an anonymized sample intended for learning and experimentation.

Several limitations should be considered:

- Client names, URLs, and keywords have been removed to protect privacy.
- The dataset represents a snapshot rather than a continuously updated production system.
- Some columns contain missing values that require careful handling.
- The dataset should be used for decision support rather than making causal claims.
- The starter dataset is much smaller than the production warehouse dataset used later in the internship.

In [5]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nData Types")

display(df.dtypes)

Rows: 30000
Columns: 44

Data Types


,0
content_id,object
client_id,object
search_volume,float64
competition,float64
competition_level,object
cpc,float64
content_type,object
main_intent,object
word_count,float64
char_count,float64


## Self-check

- [x] I defined the unit of analysis.
- [x] I described the time window.
- [x] I separated features, labels, context, and excluded fields.
- [x] I verified my assumptions using code.
- [x] I documented important limitations of the dataset.
- [x] I explained why some fields should not be used for modeling.